# 01 — Camada de dados (DAL)

Desenvolve as funções de acesso a dados (baixar, gravar/ler SQLite, retornos). **F1, NF6.**

In [2]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import sqlite3
from contextlib import closing
from typing import Sequence
import pandas as pd
print('setup OK')

setup OK


## Desenvolvimento

As funções abaixo foram escritas aqui e, após os testes, movidas para `app/dal.py`.

In [3]:
def _validar_identificador(nome: str) -> None:
    """Impede nomes de tabela inválidos/injeção (o nome vem do código, mas
    validar mantém o contrato explícito)."""
    if not nome.isidentifier():
        raise ValueError(f"nome de tabela inválido: {nome!r}")

In [4]:
# Frequencias suportadas: a granularidade da coluna ``data`` segue a frequencia.
FORMATO_DATA = {"1mo": "%Y-%m", "1d": "%Y-%m-%d"}


def _formato_data(frequencia: str) -> str:
    """Formato da coluna ``data`` para a frequencia pedida (valida o argumento)."""
    try:
        return FORMATO_DATA[frequencia]
    except KeyError:
        raise ValueError(
            f"frequência desconhecida: {frequencia!r} (use {sorted(FORMATO_DATA)})."
        ) from None

In [5]:
def baixar_precos(
    ativos: Sequence[str],
    inicio: str,
    fim: str | None = None,
    frequencia: str = "1mo",
) -> pd.DataFrame:
    """Baixa preços de fechamento (ajustado) do Yahoo Finance. (F1)

    Parameters
    ----------
    ativos : sequência de tickers, ex.: ``["^BVSP"]``.
    inicio : data inicial ``AAAA-MM-DD``.
    fim : data final ``AAAA-MM-DD`` (``None`` => hoje).
    frequencia : ``"1mo"`` (mensal) ou ``"1d"`` (diário).

    Returns
    -------
    DataFrame com a coluna ``data`` na primeira posição — ``AAAA-MM`` se mensal,
    ``AAAA-MM-DD`` se diário — e uma coluna por ticker (fechamento ajustado).

    Notes
    -----
    Usa a *chart API* pública do Yahoo via ``urllib`` (stdlib) — **sem**
    ``yfinance``/``curl_cffi``, que sofriam de erro de certificado SSL no
    Windows. Imports tardios mantêm os testes de banco/retorno offline (NF4).
    """
    import json
    from datetime import datetime, timezone
    from urllib.parse import quote
    from urllib.request import Request, urlopen

    fmt = _formato_data(frequencia)

    def _epoch(d: str) -> int:
        return int(pd.Timestamp(d, tz="UTC").timestamp())

    p1 = _epoch(inicio)
    p2 = _epoch(fim or datetime.now(timezone.utc).strftime("%Y-%m-%d"))

    series: dict[str, pd.Series] = {}
    for tk in ativos:
        url = (f"https://query1.finance.yahoo.com/v8/finance/chart/{quote(tk)}"
               f"?period1={p1}&period2={p2}&interval={frequencia}") 
        req = Request(url, headers={"User-Agent": "Mozilla/5.0"}) 
        with urlopen(req, timeout=30) as resp: 
            payload = json.load(resp) 
        resultado = (payload.get("chart") or {}).get("result") 
        if not resultado:
            raise ValueError(f"Yahoo não retornou dados para {tk!r}.")
        res = resultado[0] 
        ts = res.get("timestamp") or [] 
        close = res["indicators"]["quote"][0].get("close") or []
        datas = [datetime.fromtimestamp(t, tz=timezone.utc).strftime(fmt) for t in ts] 
        series[tk] = pd.Series(close, index=datas, name=tk) 

    df = pd.DataFrame(series).dropna(how="any")
    return df.reset_index().rename(columns={"index": "data"})

In [6]:
# Serie do CDI na SGS por frequencia: 4391 = acumulada no mes (% a.m.) · 12 = (% a.d.)
_SERIE_CDI = {"1mo": 4391, "1d": 12}
# A SGS recusa (HTTP 406) series DIARIAS de mais de 10 anos por requisicao.
_LIMITE_ANOS_SGS = {"1mo": None, "1d": 10}
# A SGS e lenta e irregular em janelas diarias largas (medido: 0,4s a 19s).
_TIMEOUT_SGS = 90


def _janelas(inicio, fim, limite_anos):
    """Fatia [inicio, fim] em janelas de no máximo ``limite_anos`` (None = 1 só)."""
    if limite_anos is None or fim <= inicio + pd.DateOffset(years=limite_anos):
        return [(inicio, fim)]
    partes, atual = [], inicio
    while atual <= fim:
        prox = min(atual + pd.DateOffset(years=limite_anos), fim)
        partes.append((atual, prox))
        atual = prox + pd.Timedelta(days=1)
    return partes


def baixar_cdi_bcb(inicio: str, fim: str | None = None,
                   frequencia: str = "1mo") -> pd.DataFrame:
    """Baixa o CDI da API SGS do Banco Central. (F1)

    Série 4391 = 'CDI acumulada no mês' (% a.m.) para ``frequencia="1mo"``;
    série 12 = 'CDI' (% a.d.) para ``"1d"``. Devolve um DataFrame com ``data``
    (``AAAA-MM`` ou ``AAAA-MM-DD``) e ``cdi`` (decimal **por período**).
    Fonte oficial, gratuita e sem cadastro; requer internet.

    Notes
    -----
    Imports tardios (``json``/``urllib``, stdlib) e nenhuma dependência nova.
    A série diária é baixada em janelas de 10 anos e concatenada.
    """
    import json
    from datetime import date as _date
    from urllib.request import urlopen

    fmt = _formato_data(frequencia)
    serie = _SERIE_CDI[frequencia]
    ini = pd.to_datetime(inicio)
    dfim = pd.to_datetime(fim or _date.today().isoformat())

    partes = []
    for janela_ini, janela_fim in _janelas(ini, dfim, _LIMITE_ANOS_SGS[frequencia]):
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
               f"?formato=json&dataInicial={janela_ini.strftime('%d/%m/%Y')}"
               f"&dataFinal={janela_fim.strftime('%d/%m/%Y')}")
        with urlopen(url, timeout=_TIMEOUT_SGS) as resp:
            partes.extend(json.load(resp))
    if not partes:
        raise ValueError("BCB não retornou CDI para o período pedido.")
    df = pd.DataFrame(partes)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y").dt.strftime(fmt)
    df["cdi"] = df["valor"].astype(float) / 100.0 
    # Janelas consecutivas podem repetir a data de fronteira.
    return df[["data", "cdi"]].drop_duplicates(subset="data").reset_index(drop=True)

In [7]:
def calcular_retornos(precos: pd.DataFrame, coluna_data: str = "data") -> pd.DataFrame:
    """Converte preços em retornos simples por período. (F1)

    A primeira observação é descartada (não há retorno anterior), portanto
    ``T_efetivo = T − 1``. Todas as colunas que não sejam ``coluna_data`` são
    tratadas como séries de preço.

    Notes
    -----
    A taxa CDI, por **já ser** um retorno (não um preço), não passa por aqui:
    ela entra direto na coluna ``cdi`` da tabela ``retornos``.
    """
    if coluna_data not in precos.columns:
        raise KeyError(f"coluna '{coluna_data}' ausente em precos.")

    datas = precos[coluna_data].iloc[1:].to_numpy()
    numericas = precos.drop(columns=[coluna_data])
    ret = numericas.pct_change(fill_method=None).iloc[1:].reset_index(drop=True)
    ret.insert(0, coluna_data, datas)
    return ret

In [8]:
def gravar_sqlite(
    df: pd.DataFrame,
    db_path: str,
    tabela: str,
    if_exists: str = "replace",
) -> None:
    """Grava um DataFrame numa tabela do banco SQLite. (F1, NF6)

    Usa ``contextlib.closing`` porque ``with sqlite3.connect(...)`` apenas
    gerencia a transação — **não fecha** a conexão; sem fechar, o arquivo do
    banco fica travado no Windows (NF3).
    """
    _validar_identificador(tabela)
    with closing(sqlite3.connect(db_path)) as con:
        df.to_sql(tabela, con, if_exists=if_exists, index=False)
        con.commit() 

In [9]:
def ler_sqlite(db_path: str, tabela: str) -> pd.DataFrame:
    """Lê uma tabela do banco SQLite para um DataFrame. (F1, NF6)"""
    _validar_identificador(tabela)
    with closing(sqlite3.connect(db_path)) as con:
        return pd.read_sql(f"SELECT * FROM {tabela}", con)


**Teste** — retornos, round-trip SQLite e downloads reais (protegidos).

In [10]:
import pandas as pd, os, tempfile
precos = pd.DataFrame({'data': ['2000-01','2000-02','2000-03','2000-04'], 
                       'ibov': [100.,102.,99.,105.]})
print('calcular_retornos:'); print(calcular_retornos(precos))

esp = precos['ibov'].pct_change(fill_method=None).iloc[1:].reset_index(drop=True)
assert calcular_retornos(precos)['ibov'].round(10).tolist() == esp.round(10).tolist()

calcular_retornos:
      data      ibov
0  2000-02  0.020000
1  2000-03 -0.029412
2  2000-04  0.060606


In [11]:
db = os.path.join(tempfile.gettempdir(), 'dev01.db'); gravar_sqlite(precos, db, 'ibovespa')
lido = ler_sqlite(db, 'ibovespa'); pd.testing.assert_frame_equal(precos.reset_index(drop=True), lido, check_dtype=False); os.remove(db)
print('round-trip SQLite: OK')

try:
    print(baixar_precos(['^BVSP'], '2023-01-01', '2023-04-01'))
    print(baixar_cdi_bcb('2023-01-01', '2023-04-01'))
except Exception as e:
    print('download offline:', type(e).__name__, e)
print('OK')

round-trip SQLite: OK
      data     ^BVSP
0  2023-01  113532.0
1  2023-02  104932.0
2  2023-03  101882.0
      data     cdi
0  2023-01  0.0112
1  2023-02  0.0092
2  2023-03  0.0117
3  2023-04  0.0092
OK


In [12]:
# --- frequencia e fatiamento de janelas (offline, deterministico) ---
from app import dal as _dal                 

assert _dal.FORMATO_DATA == {'1mo': '%Y-%m', '1d': '%Y-%m-%d'}
assert _dal._formato_data('1mo') == '%Y-%m' and _dal._formato_data('1d') == '%Y-%m-%d'

In [13]:
# a SGS recusa (HTTP 406) series diarias de mais de 10 anos por requisicao
j_curta = _dal._janelas(pd.Timestamp('2022-05-22'), pd.Timestamp('2026-08-06'), 10)
j_longa = _dal._janelas(pd.Timestamp('2000-01-01'), pd.Timestamp('2026-08-06'), 10)
j_sem   = _dal._janelas(pd.Timestamp('2000-01-01'), pd.Timestamp('2026-08-06'), None)
print('janelas: curta=%d longa=%d sem_limite=%d' % (len(j_curta), len(j_longa), len(j_sem)))
assert len(j_curta) == 1 and len(j_sem) == 1 and len(j_longa) == 3
assert j_longa[0][0] == pd.Timestamp('2000-01-01') and j_longa[-1][1] == pd.Timestamp('2026-08-06')
assert all(a <= b for a, b in j_longa)                                    # cada janela é valida
assert all(j_longa[k][1] < j_longa[k+1][0] for k in range(len(j_longa)-1))  # e nao se sobrepoem
print('F1 frequencia/janelas: PASSOU')

janelas: curta=1 longa=3 sem_limite=1
F1 frequencia/janelas: PASSOU
